In [3]:
import pandas as pd
import requests
import pathlib

DATA_DIR = r"C:\Users\brand\OneDrive\Imágenes\Escritorio\Git Portfolio\bank-marketing-mlop\data\test"

X_test = pd.read_csv(DATA_DIR + r"\X_test.csv")

# Convert all columns to native Python type — fixes int64/bool serialization
X_test = X_test.astype(float)

# ── Test /health endpoint ────────────────────────────────────────────
response = requests.get('http://127.0.0.1:8000/health')
print('Health check:', response.json())

# ── Test /predict endpoint — single row ─────────────────────────────
sample_row = X_test.iloc[0].tolist()
payload = {'features': sample_row}

response = requests.post('http://127.0.0.1:8000/predict', json=payload)
print('Status code:', response.status_code)
print('Prediction:', response.json())

# ── Batch test: 100 rows ─────────────────────────────────────────────
results = []
for i in range(100):
    row = X_test.iloc[i].tolist()
    r = requests.post('http://127.0.0.1:8000/predict', json={'features': row})
    if r.status_code == 200:
        results.append(r.json())
    else:
        print(f"Row {i} failed: {r.status_code} — {r.text}")

preds  = [r['predicted_class'] for r in results]
probas = [r['probability_subscribe'] for r in results]

print(f'Subscriptions predicted: {sum(preds)} / {len(results)}')
print(f'Avg probability: {sum(probas)/len(probas):.4f}')

Health check: {'status': 'ok', 'model': 'BankMarketingClassifier', 'version': '1'}
Status code: 200
Prediction: {'predicted_class': 0, 'probability_subscribe': 0.0591}
Subscriptions predicted: 5 / 100
Avg probability: 0.1271


## Endpoint Validation — Interpretation

Health check confirms the model is correctly loaded and served via FastAPI.

Batch results on 100 test rows:
- 5% predicted subscription rate vs ~11% true base rate
- Average probability of 0.127 reflects appropriately low confidence
  on a heavily imbalanced dataset

The gap between predicted rate (5%) and true rate (11%) is consistent
with Recall of 0.491 from training — the model misses roughly half of
true positives at the default 0.5 decision threshold.

For a production deployment, threshold tuning would be the next step:
lowering the decision threshold from 0.5 to ~0.3 would increase the
predicted positive rate closer to the true base rate, trading some
precision for higher recall — appropriate for a marketing campaign
where missing a subscriber is more costly than a wasted call.